# 🏗️ Hands-on: Pipeline ELT com Delta Lake
**Programa Lighthouse — Módulo 2: Introdução ao Databricks**

---
### Objetivo
Construir um pipeline ELT completo usando dados reais da API Open-Meteo (previsão do tempo — São Paulo).

| Camada | O que acontece |
|--------|----------------|
| **Bronze** | Dado bruto da API → Delta Table (imutável) |
| **Silver** | Tipagem + colunas enriquecidas + particionamento por data |
| **Gold**   | Agregação diária para consumo analítico |

---
> ⚠️ **Antes de começar:** substitua `SEU_NOME` na célula de setup pelo seu nome (sem espaços ou acentos).


## ⚙️ Setup


In [0]:
# 🔧 ALTERE AQUI: coloque seu nome (sem espaços ou acentos)
SEU_NOME = "seu_nome"   # ex: "guilherme", "ana", "pedro"

CAT_RAW  = "aularaw"
CAT_PROD = "aulaproducao"
SCHEMA   = "seu_nome"

TABELA_BRONZE = f"{CAT_RAW}.{SCHEMA}.previsao"
TABELA_SILVER = f"{CAT_PROD}.{SCHEMA}.previsao_silver"
TABELA_GOLD   = f"{CAT_PROD}.{SCHEMA}.previsao_gold"

print(f"✅ Bronze : {TABELA_BRONZE}")
print(f"   Silver : {TABELA_SILVER}")
print(f"   Gold   : {TABELA_GOLD}")


✅ Bronze : aularaw.sara.previsao
   Silver : aulaproducao.sara.previsao_silver
   Gold   : aulaproducao.sara.previsao_gold


### 🗂️ Criar Catálogos e Schemas

A aula usa dois catálogos separados para representar as camadas:

| Catálogo | Schema | Camada | Papel |
|----------|--------|--------|-------|
| `aularaw` | `<seu_nome>` | Bronze | Dado bruto, imutável, append-only |
| `aulaproducao` | `<seu_nome>` | Silver + Gold | Dado transformado e curado |

> 💡 Os catálogos geralmente são criados pelo **instrutor**. Cada aluno cria o próprio schema dentro deles.


In [0]:
# Catálogos da aula (criados pelo instrutor — rode só se tiver permissão de admin)
spark.sql('CREATE CATALOG IF NOT EXISTS aularaw')
spark.sql('CREATE CATALOG IF NOT EXISTS aulaproducao')

# Schema pessoal do aluno em cada catálogo
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CAT_RAW}.{SCHEMA}')
spark.sql(f'CREATE SCHEMA IF NOT EXISTS {CAT_PROD}.{SCHEMA}')

print('=== Schemas em aularaw ===')
display(spark.sql('SHOW SCHEMAS IN aularaw'))


=== Schemas em aularaw ===


databaseName
default
information_schema
sara


## 🪄 Passo 1 — Magic Commands e dbutils

O Databricks tem atalhos chamados **magic commands** para trocar a linguagem de uma célula:

| Comando | O que faz |
|---------|-----------|
| `%sql`  | Executa SQL na célula |
| `%md`   | Renderiza Markdown |
| `%sh`   | Shell (bash) |
| `%run`  | Executa outro notebook |
| `%fs`   | Atalho para `dbutils.fs` |


In [0]:
# dbutils: utilitários nativos do Databricks
# Listar o Data Lake (DBFS)
display(dbutils.fs.ls('dbfs:/'))


path,name,size,modificationTime
dbfs:/Volumes/,Volumes/,0,0
dbfs:/Workspace/,Workspace/,0,0
dbfs:/databricks-datasets/,databricks-datasets/,0,0


In [0]:
# %sql é um magic command — troca a linguagem da célula para SQL
%sql
SELECT current_user()    AS usuario,
       current_catalog() AS catalogo,
       current_database() AS schema_atual


## 📥 Passo 2 — Ler Dados da API (Camada Bronze)

Usaremos a **Open-Meteo API** — gratuita, sem autenticação, sem API key.
Ela retorna previsão horária de temperatura, precipitação e vento para São Paulo (7 dias = 168 linhas).


In [0]:
import requests
import pandas as pd

# ── Parâmetros da API ────────────────────────────────────────────────
URL = "https://api.open-meteo.com/v1/forecast"
PARAMS = {
    "latitude"     : -23.55,
    "longitude"    : -46.63,
    "hourly"       : "temperature_2m,precipitation,windspeed_10m,weathercode",
    "forecast_days": 7,
    "timezone"     : "America/Sao_Paulo",
}

# ── Chamada HTTP ─────────────────────────────────────────────────────
response = requests.get(URL, params=PARAMS)
response.raise_for_status()   # lança erro se status != 200
data = response.json()

print(f"Status      : {response.status_code}")
print(f"Localização : {data['latitude']}°S, {data['longitude']}°W")
print(f"Timezone    : {data['timezone']}")
print(f"Períodos    : {len(data['hourly']['time'])} horas")


Status      : 200
Localização : -23.514938°S, -46.610504°W
Timezone    : America/Sao_Paulo
Períodos    : 168 horas


In [0]:
# ── Montar DataFrame Pandas ──────────────────────────────────────────
df_pandas = pd.DataFrame({
    "datetime"     : data["hourly"]["time"],
    "temperatura"  : data["hourly"]["temperature_2m"],
    "precipitacao" : data["hourly"]["precipitation"],
    "vento_kmh"    : data["hourly"]["windspeed_10m"],
    "codigo_tempo" : data["hourly"]["weathercode"],
})

print(f"Shape: {df_pandas.shape}   (linhas, colunas)")
df_pandas.head(5)


Shape: (168, 5)   (linhas, colunas)


,datetime,temperatura,precipitacao,vento_kmh,codigo_tempo
0,2026-05-26T00:00,16.5,0.0,6.0,3
1,2026-05-26T01:00,16.3,0.0,6.4,3
2,2026-05-26T02:00,16.3,0.0,4.7,3
3,2026-05-26T03:00,16.4,0.0,4.6,3
4,2026-05-26T04:00,16.2,0.0,4.5,3


## 🔥 Passo 3 — Pandas → Spark DataFrame

O Pandas carrega tudo na **memória do driver** — não distribui.
Convertendo para Spark, habilitamos processamento distribuído e escrita Delta nativa.

```
Pandas   →  memória RAM do driver  →  sem paralelismo
Spark    →  distribuído nos workers →  escala para TB
```


In [0]:
# spark = sessão Spark disponível automaticamente no Databricks
df_bronze = spark.createDataFrame(df_pandas)

print("=== Schema inferido ===")
df_bronze.printSchema()
print(f"\n=== Total: {df_bronze.count()} linhas ===")
display(df_bronze.limit(5))


=== Schema inferido ===
root
 |-- datetime: string (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- precipitacao: double (nullable = true)
 |-- vento_kmh: double (nullable = true)
 |-- codigo_tempo: long (nullable = true)


=== Total: 168 linhas ===


datetime,temperatura,precipitacao,vento_kmh,codigo_tempo
2026-05-26T00:00,16.5,0.0,6.0,3
2026-05-26T01:00,16.3,0.0,6.4,3
2026-05-26T02:00,16.3,0.0,4.7,3
2026-05-26T03:00,16.4,0.0,4.6,3
2026-05-26T04:00,16.2,0.0,4.5,3


## 💾 Passo 4 — Salvar Camada Bronze


In [0]:

# Salvar como Delta Table
(df_bronze
    .write
    .format("delta")
    .mode("overwrite")       # em prod: "append" para preservar histórico
    .saveAsTable(TABELA_BRONZE)
)

print(f"✅ Bronze salva: {TABELA_BRONZE}")


✅ Bronze salva: aularaw.sara.previsao


## 🥈 Passo 5 — Transformar: Bronze → Silver

Na camada Silver fazemos:
- **Tipagem correta**: `datetime` string → timestamp, extração de `data` e `hora`
- **Enriquecimento**: classificação de temperatura + descrição do código WMO
- **Particionamento**: `partitionBy('data')` — cada dia = uma pasta no storage


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from itertools import chain

# Mapa de códigos WMO → descrição legível (subset dos mais comuns)
WMO_MAP = {
    0: 'Céu limpo',          1: 'Predominantemente limpo', 2: 'Parcialmente nublado',
    3: 'Nublado',            45: 'Neblina',                51: 'Garoa leve',
    61: 'Chuva leve',        63: 'Chuva moderada',         65: 'Chuva intensa',
    80: 'Pancadas leves',    81: 'Pancadas moderadas',     95: 'Trovoada',
}

# Broadcast para todos os workers
wmo_spark_map = F.create_map([F.lit(x) for x in chain(*WMO_MAP.items())])  # ✅ nativo Spark


In [0]:
# ── Transformações Bronze → Silver ──────────────────────────────────
df_silver = (
    spark.read.table(TABELA_BRONZE)         # ler da Bronze
    .withColumn('datetime',   F.to_timestamp('datetime'))
    .withColumn('data',       F.to_date('datetime'))
    .withColumn('hora',       F.hour('datetime'))
    .withColumn('classificacao',
        F.when(F.col('temperatura') >= 35, 'Muito Quente')
         .when(F.col('temperatura') >= 28, 'Quente')
         .when(F.col('temperatura') >= 18, 'Ameno')
         .when(F.col('temperatura') >= 10, 'Frio')
         .otherwise('Muito Frio')
    )
    .withColumn("condicao_tempo",
    F.coalesce(
        wmo_spark_map[F.col("codigo_tempo")],
        F.concat(F.lit("Código "), F.col("codigo_tempo").cast("string"))
    )
    )
)
print("=== Schema Silver ===")
df_silver.printSchema()
display(df_silver.limit(10))


=== Schema Silver ===
root
 |-- datetime: timestamp (nullable = true)
 |-- temperatura: double (nullable = true)
 |-- precipitacao: double (nullable = true)
 |-- vento_kmh: double (nullable = true)
 |-- codigo_tempo: long (nullable = true)
 |-- data: date (nullable = true)
 |-- hora: integer (nullable = true)
 |-- classificacao: string (nullable = false)
 |-- condicao_tempo: string (nullable = true)



datetime,temperatura,precipitacao,vento_kmh,codigo_tempo,data,hora,classificacao,condicao_tempo
2026-05-26T00:00:00.000Z,16.5,0.0,6.0,3,2026-05-26,0,Frio,Nublado
2026-05-26T01:00:00.000Z,16.3,0.0,6.4,3,2026-05-26,1,Frio,Nublado
2026-05-26T02:00:00.000Z,16.3,0.0,4.7,3,2026-05-26,2,Frio,Nublado
2026-05-26T03:00:00.000Z,16.4,0.0,4.6,3,2026-05-26,3,Frio,Nublado
2026-05-26T04:00:00.000Z,16.2,0.0,4.5,3,2026-05-26,4,Frio,Nublado
2026-05-26T05:00:00.000Z,16.1,0.0,6.8,3,2026-05-26,5,Frio,Nublado
2026-05-26T06:00:00.000Z,16.2,0.0,7.7,3,2026-05-26,6,Frio,Nublado
2026-05-26T07:00:00.000Z,16.3,0.0,8.0,3,2026-05-26,7,Frio,Nublado
2026-05-26T08:00:00.000Z,16.7,0.0,8.2,3,2026-05-26,8,Frio,Nublado
2026-05-26T09:00:00.000Z,18.3,0.0,7.7,3,2026-05-26,9,Ameno,Nublado


In [0]:
# Salvar Silver — particionada por data
(df_silver
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy('data')     # cada data = uma pasta no S3/ADLS
    .saveAsTable(TABELA_SILVER)
)

print(f"✅ Silver salva: {TABELA_SILVER}")
print(f"   Partições (dias): {df_silver.select('data').distinct().count()}")


✅ Silver salva: aulaproducao.sara.previsao_silver
   Partições (dias): 7


## 🔍 Passo 6 — Queries SQL na Silver

> Você pode misturar células Python e SQL no mesmo notebook usando `%sql`.


In [0]:
# Opção A: Python com createOrReplaceTempView
df_silver.createOrReplaceTempView('vw_silver')

resultado = spark.sql("""
    SELECT
        data,
        classificacao,
        COUNT(*)                       AS horas,
        ROUND(AVG(temperatura), 1)     AS temp_media,
        ROUND(MIN(temperatura), 1)     AS temp_min,
        ROUND(MAX(temperatura), 1)     AS temp_max,
        ROUND(SUM(precipitacao), 1)    AS chuva_total_mm
    FROM vw_silver
    GROUP BY 1, 2
    ORDER BY 1, temp_media DESC
""")

display(resultado)


data,classificacao,horas,temp_media,temp_min,temp_max,chuva_total_mm
2026-05-26,Ameno,15,21.4,18.3,24.3,0.1
2026-05-26,Frio,9,16.3,16.1,16.7,0.0
2026-05-27,Ameno,10,20.2,18.1,23.0,1.4
2026-05-27,Frio,14,16.5,14.9,17.9,0.1
2026-05-28,Ameno,5,18.5,18.0,19.0,0.0
2026-05-28,Frio,19,13.9,11.7,17.1,0.0
2026-05-29,Ameno,5,19.1,18.1,19.8,0.0
2026-05-29,Frio,19,13.8,12.2,17.8,0.0
2026-05-30,Frio,24,14.6,12.1,17.8,0.0
2026-05-31,Ameno,8,20.3,18.5,21.6,0.0


## ⏳ Passo 7 — DESCRIBE HISTORY e Time Travel

O Delta Lake mantém um **log de transações** (`_delta_log`) com todas as versões da tabela.
Isso habilita o **Time Travel**: consultar como o dado estava em qualquer versão anterior.


In [0]:
%sql


DESCRIBE HISTORY aulaproducao.seu_nome.previsao_silver


version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
0,2026-05-26T15:53:10.000Z,73920920072922,ana.cunha@indicium.tech,CREATE OR REPLACE TABLE AS SELECT,"Map(partitionBy -> [""data""], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(4391573717819732),2374969c-6de9-430d-b9c2-1cce56aaab32,0526-154726-zte0md0u-v2n,null,WriteSerializable,false,"Map(numFiles -> 7, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 168, numOutputBytes -> 22215)",null,Databricks-Runtime/18.1.x-aarch64-photon-scala2.13


In [0]:
%sql

SELECT * FROM aulaproducao.seu_nome.previsao_silver
VERSION AS OF 0
LIMIT 5


datetime,temperatura,precipitacao,vento_kmh,codigo_tempo,data,hora,classificacao,condicao_tempo
2026-05-29T00:00:00.000Z,13.4,0.0,6.8,3,2026-05-29,0,Frio,Nublado
2026-05-29T01:00:00.000Z,13.4,0.0,5.8,3,2026-05-29,1,Frio,Nublado
2026-05-29T02:00:00.000Z,13.4,0.0,6.1,3,2026-05-29,2,Frio,Nublado
2026-05-29T03:00:00.000Z,13.3,0.0,6.9,3,2026-05-29,3,Frio,Nublado
2026-05-29T04:00:00.000Z,12.9,0.0,7.7,3,2026-05-29,4,Frio,Nublado


In [0]:
# Time Travel via Python (útil em pipelines)
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .table(TABELA_SILVER)
)
print(f"Versão 0 — linhas: {df_v0.count()}")


Versão 0 — linhas: 168


## 🏅 Passo 8 (Bônus) — Camada Gold

A Gold é otimizada para **BI e relatórios executivos**: dados agregados, prontos para dashboard.


In [0]:
df_gold = (
    spark.read.table(TABELA_SILVER)
    .groupBy("data")
    .agg(
        F.round(F.avg("temperatura"), 1).alias("temp_media"),
        F.round(F.min("temperatura"), 1).alias("temp_min"),
        F.round(F.max("temperatura"), 1).alias("temp_max"),
        F.round(F.sum("precipitacao"), 1).alias("chuva_total_mm"),
        F.round(F.avg("vento_kmh"), 1).alias("vento_medio_kmh"),
    )
    .orderBy("data")
)

(df_gold.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(TABELA_GOLD)
)

print(f"✅ Gold salva: {TABELA_GOLD}")
display(df_gold)

✅ Gold salva: aulaproducao.sara.previsao_gold


data,temp_media,temp_min,temp_max,chuva_total_mm,vento_medio_kmh
2026-05-26,19.5,16.1,24.3,0.1,6.2
2026-05-27,18.1,14.9,23.0,1.5,7.4
2026-05-28,14.9,11.7,19.0,0.0,7.0
2026-05-29,14.9,12.2,19.8,0.0,8.5
2026-05-30,14.6,12.1,17.8,0.0,6.8
2026-05-31,16.3,12.5,21.6,0.0,4.4
2026-06-01,15.9,13.1,21.1,0.0,2.9


## 🔎 Passo 9 — Explorar no Data Explorer (Unity Catalog)

1. Clique em **Data** no menu lateral
2. Navegue até `main` → `<seu_nome>` → `previsao_silver`
3. Explore as abas: **Schema**, **Sample Data**, **Detail**, **History**, **Permissions**


In [0]:
%sql
DESCRIBE EXTENDED aulaproducao.seu_nome.previsao_silver


col_name,data_type,comment
datetime,timestamp,null
temperatura,double,null
precipitacao,double,null
vento_kmh,double,null
codigo_tempo,bigint,null
data,date,null
hora,int,null
classificacao,string,null
condicao_tempo,string,null
# Partition Information,,


## 🧹 Limpeza (opcional)

O `VACUUM` remove arquivos físicos antigos do Delta Log.
Por padrão, respeita **7 dias de retenção** para não quebrar o Time Travel.

> ⚠️ Nunca execute em produção sem entender o impacto no Time Travel.


In [0]:
%sql

-- VACUUM aulaproducao.seu_nome.previsao_silver RETAIN 168 HOURS DRY RUN


path
""


## ✅ Resumo

```
API Open-Meteo
     │
     ▼
[Bronze]  previsao_bronze  ← dado bruto, string, sem transformação
     │
     ▼  tipagem + enriquecimento + partitionBy('data')
[Silver]  previsao_silver  ← fonte da verdade, pronta para análise
     │
     ▼  agregação diária
[Gold]    previsao_gold    ← pronta para dashboard / BI
```

### Conceitos praticados
| Conceito | Onde |
|----------|------|
| Magic commands (`%sql`, `%md`) | Passo 1 |
| `dbutils.fs` | Passo 1 |
| Pandas → Spark (`createDataFrame`) | Passo 3 |
| Delta Lake: `saveAsTable`, `partitionBy` | Passos 4 e 5 |
| PySpark: `withColumn`, `F.when`, UDF | Passo 5 |
| SQL no notebook (`%sql` e `spark.sql`) | Passo 6 |
| `DESCRIBE HISTORY` + Time Travel | Passo 7 |
| Arquitetura Medallion Bronze/Silver/Gold | Todo o notebook |


In [0]:
%sql
DROP CATALOG IF EXISTS aulalh      CASCADE;
DROP CATALOG IF EXISTS aulaproducao CASCADE;
DROP CATALOG IF EXISTS aularaw      CASCADE;